In [ ]:
'''
1、调用 API 访问大模型（以 Deepseek 为例）
'''


from openai import OpenAI

ds = {
    "api_key": "sk-54c8ff13fe1b482eb667c6d13e2f2df7",
    "base_url": "https://api.deepseek.com",
}

def deepseek_stream(prompt: str, llm: dict, model: str = "deepseek-flash"):
    """
    调用 Deepseek API 进行流式对话
    prompt: 用户输入的提示词
    llm: 包含 API Key 和 Base URL 的字典
    model: 使用的模型名称
    """

    # 1、初始化客户端
    client = OpenAI(api_key=llm["api_key"], base_url=llm["base_url"])

    # 2、创建流式请求
    stream = client.chat.completions.create(
        model=model,  # 指定使用的模型
        messages=[
            {"role": "assistant", "content": "You are a helpful C language coding assistant. Use Chinese to answer questions."},
            {"role": "user", "content": prompt}
        ],  # 消息列表
        stream=True,  # 启用流式响应，默认为 False, 渐进输出，而不是直接输出全部
        stream_options={
            "max_tokens": 1000,  # 最大输出 token 数量
            "include_usage": True  # 是否包含token使用情况统计信息
        },
        reasoning_effort="high",  # 推理努力程度(根据对应模型官方提供的选项确定)，可选值：low、medium、high
        extra_body={"thinking": {"type": "enabled"}}  # 启用思考模式
    )

    # 3、处理流式响应
    for chunk in stream:
        if chunk.usage is not None:
            # 什么时候不为空？
            # 通常在回复结束时，Deepseek 会返回 token 使用情况统计信息。前面的 chunk 中都是 None
            print(f"Token usage: {chunk.usage}")

        if not chunk.choices:
            continue

        # .delta 属性包含了思考内容和输出内容, .delta 表示增量输出, 即每次输出的内容
        choice = chunk.choices[0].delta
        # 分别获取思考内容和输出内容
        '''
            上面已经开启了思考模式，并且流式传输，每次响应都是 delta 增量输出，delta 中包含了思考内容和输出内容。
            通常在前部分是思考内容，即 reasoning_content, 后部分是输出内容, 即 content。
            例如: 此时是正在输出最终结果的部分: content 有内容, reasoning_content 为空。
                ChoiceDelta(content=' C', function_call=None, refusal=None, role=None, tool_calls=None, reasoning_content=None)
        '''
        reasoning_content = getattr(choice, "reasoning_content", None)
        output_content = getattr(choice, "content", None)

        if reasoning_content:
            print(reasoning_content, end="", flush=True)

        if output_content:
            print(output_content, end="", flush=True)


def deepseek_no_stream(prompt: str, llm: dict, model: str = "deepseek-flash"):
    """
    调用 Deepseek API 进行非流式对话，一次性输出全部结果
    """

    # 1、初始化客户端
    client = OpenAI(api_key=llm["api_key"], base_url=llm["base_url"])

    # 2、创建非流式请求
    response = client.chat.completions.create(
        model=model,  # 指定使用的模型
        messages=[
            {"role": "assistant", "content": "You are a helpful C language coding assistant. Use Chinese to answer questions."},
            {"role": "user", "content": prompt}
        ],  # 消息列表
        stream=False,  # 非流式响应，默认为 False, 一次性输出全部结果
        reasoning_effort="high",  # 推理努力程度(根据对应模型官方提供的选项确定)，可选值：low、medium、high
        extra_body={"thinking": {"type": "enabled"}}  # 启用思考模式
    )

    # 3、处理非流式响应
    if response.usage is not None:
        print(f"Token usage: {response.usage}")

    if not response.choices:
        return

    # 非流式对话中，response.choices[0] 是一个完整的 Choice 对象，而不是增量输出的 ChoiceDelta 对象。
    # 可以直接获取 choices[0] 的 message 属性，该属性包含了思考内容和输出内容。
    choice = response.choices[0].message
    print(f"Choice: {choice}")
    reasoning_content = getattr(choice, "reasoning_content", None)
    output_content = getattr(choice, "content", None)

    if reasoning_content:
        print(reasoning_content)

    if output_content:
        print(output_content)

if __name__ == "__main__":
    # deepseek_stream("请帮我写一个 C 语言程序，实现两个整数的加法。", ds)
    deepseek_no_stream("请帮我写一个 C 语言程序，实现两个整数的加法。", ds)


'''
role 字段说明
role	        中文含义	        作用
system	        系统角色	    设置模型的行为规则、身份等
developer	    开发者角色	    设置应用程序的指令（OpenAI 支持，DeepSeek 兼容情况需看具体接口）
user	        用户角色	    用户向模型提出问题
assistant	    助手角色	    模型之前生成的回答
tool	        工具角色	    工具调用返回的结果

OpenAI 客户端
│
└── chat
    └── completions
        └── create(...)
            │
            ├── model             模型名称
            ├── messages          对话消息
            ├── stream            是否流式输出
            ├── stream_options    流式选项
            ├── reasoning_effort  推理强度
            │
            ▼
          DeepSeek API
            │
            ▼
          stream
            │
            ├── chunk 1
            │    └── choices[0].delta
            │         └── reasoning_content
            │
            ├── chunk 2
            │    └── choices[0].delta
            │         └── reasoning_content
            │
            ├── chunk 3
            │    └── choices[0].delta
            │         └── content
            │
            ├── chunk 4
            │    └── choices[0].delta
            │         └── content
            │
            └── 最后一个 chunk
                 ├── choices[0].finish_reason
                 └── usage
                      ├── prompt_tokens
                      ├── completion_tokens
                      └── total_tokens

每个 chunk 结构（不是每个 chunk 都有以下每个字段的信息）：
{
    "id": "string",
    "model": "string",
    "choices": [
        {
            "delta": {
                "reasoning_content": "string",
                "content": "string"
            },
            "finish_reason": "string"
        }
    ],
    "usage": {
        "prompt_tokens": 0,
        "completion_tokens": 0,
        "total_tokens": 0
    }
}

'''

Token usage: CompletionUsage(completion_tokens=225, prompt_tokens=63, total_tokens=288, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=45, rejected_prediction_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cache_write_tokens=None, cached_tokens=0), prompt_cache_hit_tokens=0, prompt_cache_miss_tokens=63)
Choice: ChatCompletionMessage(content='下面是一个实现两个整数相加的 C 语言程序：\n\n```c\n#include <stdio.h>\n\nint main(void) {\n    int a, b, sum;\n\n    printf("请输入两个整数：");\n    scanf("%d %d", &a, &b);\n\n    sum = a + b;\n\n    printf("%d + %d = %d\\n", a, b, sum);\n\n    return 0;\n}\n```\n\n运行示例：\n\n```text\n请输入两个整数：12 34\n12 + 34 = 46\n```\n\n说明：\n\n- `scanf("%d %d", &a, &b);` 用于读取两个整数。\n- `sum = a + b;` 计算它们的和。\n- `printf` 输出计算结果。\n\n如果需要支持更大的整数，可以把 `int` 改为 `long long`，并使用 `%lld` 格式符。', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning_cont

## 2、重要参数

`client.chat.completions.create()` 用于向大模型发送对话请求，并获取模型生成的结果。

| 参数 | 作用 | 示例 |
| --- | --- | --- |
| `model` | 指定使用的大模型 | `"deepseek-flash"` |
| `messages` | 输入消息，包括系统指令、用户问题及对话历史 | `[{...}]` |
| `stream` | 是否流式输出，默认 False | `True` / `False` |
| `max_tokens` | 限制模型最多生成多少 Token，包括推理 Token（如适用） | `1024`、`2048` |
| `temperature` | 控制输出随机性，越低越稳定，越高越随机 | `0～2` |
| `top_p` | 核采样，从累计概率达到指定阈值的候选 Token 中采样 | `0～1` |
| `top_k` | 只从概率最高的 K 个候选 Token 中采样 | `10`、`40`、`50` |
| `repetition_penalty` | 重复惩罚，降低重复生成相同内容的倾向 | `1.0`、`1.1`、`1.2` |
| `seed` | 随机种子，用于提高重复请求结果的一致性，不保证完全相同 | `42`、`1234` |
| `reasoning_effort` | 控制支持该参数的模型的推理强度 | `"low"`、`"high"` |
| `response_format` | 指定输出格式，例如 JSON | `{"type":"json_object"}` |
| `tools` | 定义模型可以调用的工具或函数 | 工具定义列表 |
| `tool_choice` | 控制模型是否调用工具 | `"auto"`、`"none"` |
| `stream_options` | 配置流式响应，例如返回 Token 统计 | `{"include_usage":True}` |

`temperature、top_p、top_k` 控制 Token `采样，repetition_penalty` 控制重复倾向，`seed` 控制随机性复现，`max_tokens` 控制生成长度。
| 参数 | 作用 | 示例 | 特点 |
| --- | --- | --- | --- |
| `temperature` | 调整概率分布，控制随机性 | `0.5` | 越低越确定，越高越随机 |
| `top_k` | 只保留概率最高的 K 个 Token | `top_k=3` | 固定候选数量，保留前 3 个 |
| `top_p` | 保留累计概率达到 P 的最小候选集合 | `top_p=0.8` | 候选数量动态变化，累计概率至少 80% |

核心区别：`temperature` 调整概率，`top_k` 限制数量，`top_p` 限制累计概率。

In [ ]:
'''
3、结构化输出
'''

def base_struct_output():
    from pydantic import BaseModel
    from openai import OpenAI

    client = OpenAI(api_key=ds["api_key"], base_url=ds["base_url"])

    resp = client.chat.completions.create(
        model="deepseek-flash",
        messages=[
            {
                "role": "system",
                "content": """
                            你是一名 C 语言编程助手，使用中文说明。
                            只输出一个 JSON 对象，不要在对象外输出其他内容或 Markdown 代码块。
                            输出格式示例：
                            {
                                "language": "c",
                                "code": "完整的 C 语言代码",
                                "explanation": "代码的中文说明"
                            }
                            """
            },
            {
                "role": "user",
                "content": "请帮我写一个 C 语言程序，实现两个整数的加法。"
            }
        ],
        response_format={"type": "json_object"},  # 开启 JSON 模式
        stream=False,
        max_tokens=1000,
        extra_body={"thinking": {"type": "disabled"}},  # 关闭思考模式
    )

    choice = resp.choices[0]
    content = choice.message.content

    print(f"Content: {content}")  # 返回 json 数据
    print(f"Choice: {choice}")

    if choice.finish_reason == "length":
        raise ValueError("输出被截断，请增大 max_tokens")

    if not content or not content.strip():
        raise ValueError("返回内容为空，请调整提示词或重试")


    class CodeEvent(BaseModel):
        language: str
        code: str
        explanation: str

    '''
        BaseModel 是 Pydantic 的基类，用于定义数据模型。
            model_validate_json() 方法用于将 JSON 字符串解析为 Pydantic 模型对象。
            model_dump_json() 方法用于将 Pydantic 模型对象序列化为 JSON 字符串。
    '''
    event = CodeEvent.model_validate_json(content) # 将 JSON 字符串解析为 CodeEvent 对象
    print(f"CodeEvent: {event.model_dump_json(ensure_ascii=False, indent=2)}") # 格式化显示 CodeEvent 对象，保留中文


def advanced_struct_output():
    import json
    from typing import List, Optional
    from pydantic import BaseModel, Field, ValidationError
    from openai import OpenAI

    # 1、定义数据模型
    class Address(BaseModel):
        street: str = Field(description="街道地址")
        city: str = Field(description="城市")
        zip_code: str = Field(description="邮政编码")
        country: str = Field(description="国家")

    class UserProfile(BaseModel):
        id: str = Field(description="用户唯一标识符")
        username: str = Field(description="用户名")
        email: str = Field(description="用户邮箱")

        full_name: Optional[str] = Field(None, description="用户全名")
        age: Optional[int] = Field(None, ge=0, le=150, description="用户年龄")

        is_active: bool = Field(description="是否活跃")
        role: str = Field(description="用户角色")

        address: Optional[Address] = Field(None, description="用户地址")
        interests: List[str] = Field(description="兴趣爱好")

        previous_addresses: Optional[List[Address]] = Field(
            None, description="历史地址"
        )

    # 2、初始化客户端
    client = OpenAI(api_key=ds["api_key"], base_url=ds["base_url"])

    # 3、获取 JSON Schema，作为模型结构输出参考，放到提示词中
    # model_json_schema() 方法用于获取 Pydantic 模型的 JSON Schema 定义，返回字典
    schema: dict = UserProfile.model_json_schema()
    print(f"UserProfile JSON Schema: {json.dumps(schema, ensure_ascii=False, indent=2)}")

    # 4、调用大模型
    response = client.chat.completions.create(
    model="deepseek-flash",
    messages=[
        {
            "role": "system",
            "content": (
                "你是一个用户信息提取助手。\n"
                "请从用户提供的文字中提取信息，输出 JSON。\n"
                "必须严格按照下面的 JSON Schema 定义字段：\n"
                f"{json.dumps(schema, ensure_ascii=False)}\n"
                "不要输出额外说明。\n"
                "无法确定的可选字段设为 null。\n"
                "无法确定的必填字段不要编造，"
                "应明确告知无法提取。"
            )
        },
        {
            "role": "user",
            "content": (
                "用户编号10001，用户名zhangsan，"
                "邮箱zhangsan@example.com，今年25岁，"
                "账号处于活跃状态，角色为admin。"
                "现住广州市天河路100号，邮编510000，中国。"
                "标签有VIP和活跃用户，"
                "兴趣爱好是编程、音乐和运动。"
            )
        }
    ],
    response_format={"type": "json_object"},
    reasoning_effort="high",
    max_tokens=2048,
    stream=False)

    # 5、处理响应
    choice = response.choices[0]
    content = choice.message.content
    print(f"Content: {content}")  # 返回 json 数据

    user = UserProfile.model_validate_json(content) # 将 JSON 字符串解析为 UserProfile 对象
    print(f"UserProfile: {user.model_dump_json(ensure_ascii=False, indent=2)}") # 格式化显示 UserProfile 对象，保留中文
    print(f"user.id: {user.id}, user.username: {user.username}, user.email: {user.email}") # 访问 UserProfile 对象的属性
    print(f"UserProfile Dict: {user.model_dump()}") # 将 UserProfile 对象转换为字典

if __name__ == "__main__":
    # base_struct_output()
    advanced_struct_output()

UserProfile JSON Schema: {
  "$defs": {
    "Address": {
      "properties": {
        "street": {
          "description": "街道地址",
          "title": "Street",
          "type": "string"
        },
        "city": {
          "description": "城市",
          "title": "City",
          "type": "string"
        },
        "zip_code": {
          "description": "邮政编码",
          "title": "Zip Code",
          "type": "string"
        },
        "country": {
          "description": "国家",
          "title": "Country",
          "type": "string"
        }
      },
      "required": [
        "street",
        "city",
        "zip_code",
        "country"
      ],
      "title": "Address",
      "type": "object"
    }
  },
  "properties": {
    "id": {
      "description": "用户唯一标识符",
      "title": "Id",
      "type": "string"
    },
    "username": {
      "description": "用户名",
      "title": "Username",
      "type": "string"
    },
    "email": {
      "description": "用户邮箱",
      "title":

## 4、提示词模板

